# ISOM5240 — Testing & Experiments Notebook

This notebook measures accuracy and runtime (CPU vs GPU) for three models, then tests the deployed Streamlit Cloud app.

## Experiment objectives
1. Model selection: compare accuracy and runtime across 3 models, on CPU and GPU.
2. App performance: test the deployed app on 3–5 YouTube videos.


## 1. Install dependencies


In [3]:
!pip install -q transformers torch pandas datasets

## 2. Import packages and load test dataset


In [2]:
import time
import numpy as np
import pandas as pd
from transformers import pipeline

# Load the test split from the prepared dataset
test_df = pd.read_csv(
    "https://raw.githubusercontent.com/chasezhang1999/youtube-emotion-analyzer/main/data/go_emotions_7class/test.csv"
)
# If the above URL does not work, download test.csv from your GitHub repo manually.

test_texts = test_df["text"].astype(str).tolist()
test_labels = test_df["label_name"].astype(str).tolist()

print(f"Loaded {len(test_texts)} test samples")
print(test_df["label_name"].value_counts().sort_index())

Loaded 2160 test samples
label_name
anger        131
disgust       76
fear          65
joy           93
neutral     1606
sadness      102
surprise      87
Name: count, dtype: int64


## 3. Define helper functions


In [4]:
# Emotion labels for the seven-class models
EMOTION_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]


def normalize_emotion(label: str) -> str:
    """Map model output label to one of our seven emotions."""
    normalized = label.strip().lower()
    if normalized in EMOTION_LABELS:
        return normalized
    return "neutral"


def measure_pipeline_runtime(model_name: str, task: str, texts: list[str], device: int) -> dict:
    """
    Measure runtime WITH and WITHOUT model loading.
    device=-1 for CPU, device=0 for GPU.
    """
    # Runtime WITH model loading
    t0 = time.perf_counter()
    pipe = pipeline(task, model=model_name, device=device)
    results = pipe(texts, truncation=True, return_token_type_ids=False)
    runtime_with_load = time.perf_counter() - t0

    # Runtime WITHOUT model loading (inference only)
    t1 = time.perf_counter()
    results_no_load = pipe(texts, truncation=True, return_token_type_ids=False)
    runtime_wo_load = time.perf_counter() - t1

    return {
        "results": results,
        "runtime_with_load_s": round(runtime_with_load, 4),
        "runtime_wo_load_s": round(runtime_wo_load, 4),
    }


def compute_emotion_accuracy(results: list[dict], true_labels: list[str]) -> float:
    """Compute accuracy by comparing predicted emotion label to true label."""
    pred_labels = [normalize_emotion(r["label"]) for r in results]
    correct = sum(p == t for p, t in zip(pred_labels, true_labels))
    return round(correct / len(true_labels), 4) if true_labels else 0.0


## 4. Experiment 1: Pre-trained Emotion Model (CPU)

Model: `j-hartmann/emotion-english-distilroberta-base`


In [5]:
MODEL_1 = "j-hartmann/emotion-english-distilroberta-base"

print(f"Measuring {MODEL_1} on CPU...")
exp1_cpu = measure_pipeline_runtime(MODEL_1, "text-classification", test_texts, device=-1)
exp1_cpu_acc = compute_emotion_accuracy(exp1_cpu["results"], test_labels)

print(f"Accuracy: {exp1_cpu_acc}")
print(f"Runtime with model loading: {exp1_cpu['runtime_with_load_s']}s")
print(f"Runtime without model loading: {exp1_cpu['runtime_wo_load_s']}s")

Measuring j-hartmann/emotion-english-distilroberta-base on CPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Accuracy: 0.6204
Runtime with model loading: 148.3473s
Runtime without model loading: 118.4442s


## 5. Experiment 1: Pre-trained Emotion Model (GPU)


In [6]:
print(f"Measuring {MODEL_1} on GPU...")
exp1_gpu = measure_pipeline_runtime(MODEL_1, "text-classification", test_texts, device=0)
exp1_gpu_acc = compute_emotion_accuracy(exp1_gpu["results"], test_labels)

print(f"Accuracy: {exp1_gpu_acc}")
print(f"Runtime with model loading: {exp1_gpu['runtime_with_load_s']}s")
print(f"Runtime without model loading: {exp1_gpu['runtime_wo_load_s']}s")

Measuring j-hartmann/emotion-english-distilroberta-base on GPU...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Accuracy: 0.6204
Runtime with model loading: 27.6285s
Runtime without model loading: 13.8853s


## 6. Experiment 2: Fine-tuned Emotion Model (CPU)

Model: your fine-tuned model on Hugging Face.
Model: `chase1zhang/youtube-emotion-distilbert`


In [7]:
MODEL_2 = "chase1zhang/youtube-emotion-distilbert"

print(f"Measuring {MODEL_2} on CPU...")
exp2_cpu = measure_pipeline_runtime(MODEL_2, "text-classification", test_texts, device=-1)
exp2_cpu_acc = compute_emotion_accuracy(exp2_cpu["results"], test_labels)

print(f"Accuracy: {exp2_cpu_acc}")
print(f"Runtime with model loading: {exp2_cpu['runtime_with_load_s']}s")
print(f"Runtime without model loading: {exp2_cpu['runtime_wo_load_s']}s")

Measuring chase1zhang/youtube-emotion-distilbert on CPU...


config.json:   0%|          | 0.00/853 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Accuracy: 0.863
Runtime with model loading: 138.9428s
Runtime without model loading: 122.6187s


## 7. Experiment 2: Fine-tuned Emotion Model (GPU)


In [8]:
print(f"Measuring {MODEL_2} on GPU...")
exp2_gpu = measure_pipeline_runtime(MODEL_2, "text-classification", test_texts, device=0)
exp2_gpu_acc = compute_emotion_accuracy(exp2_gpu["results"], test_labels)

print(f"Accuracy: {exp2_gpu_acc}")
print(f"Runtime with model loading: {exp2_gpu['runtime_with_load_s']}s")
print(f"Runtime without model loading: {exp2_gpu['runtime_wo_load_s']}s")

Measuring chase1zhang/youtube-emotion-distilbert on GPU...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Accuracy: 0.863
Runtime with model loading: 11.8892s
Runtime without model loading: 10.9848s


## 8. Experiment 3: Sentiment Model (CPU)

Model: `cardiffnlp/twitter-roberta-base-sentiment-latest`

Note: The sentiment model predicts `positive`/`neutral`/`negative`, not our seven emotions.
Accuracy for this model measures how well it classifies sentiment.


In [9]:
MODEL_3 = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# Map our seven emotions to sentiment for comparison
EMOTION_TO_SENTIMENT = {
    "joy": "positive",
    "surprise": "positive",
    "anger": "negative",
    "disgust": "negative",
    "fear": "negative",
    "sadness": "negative",
    "neutral": "neutral",
}
test_sentiments = [EMOTION_TO_SENTIMENT[label] for label in test_labels]

print(f"Measuring {MODEL_3} on CPU...")
exp3_cpu = measure_pipeline_runtime(MODEL_3, "sentiment-analysis", test_texts, device=-1)

# Compute sentiment accuracy
def compute_sentiment_accuracy(results: list[dict], true_sentiments: list[str]) -> float:
    pred_sentiments = [r["label"].strip().lower() for r in results]
    correct = sum(p == t for p, t in zip(pred_sentiments, true_sentiments))
    return round(correct / len(true_sentiments), 4) if true_sentiments else 0.0

exp3_cpu_acc = compute_sentiment_accuracy(exp3_cpu["results"], test_sentiments)

print(f"Sentiment accuracy: {exp3_cpu_acc}")
print(f"Runtime with model loading: {exp3_cpu['runtime_with_load_s']}s")
print(f"Runtime without model loading: {exp3_cpu['runtime_wo_load_s']}s")

Measuring cardiffnlp/twitter-roberta-base-sentiment-latest on CPU...


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Sentiment accuracy: 0.5444
Runtime with model loading: 244.9532s
Runtime without model loading: 241.3229s


## 9. Experiment 3: Sentiment Model (GPU)


In [10]:
print(f"Measuring {MODEL_3} on GPU...")
exp3_gpu = measure_pipeline_runtime(MODEL_3, "sentiment-analysis", test_texts, device=0)
exp3_gpu_acc = compute_sentiment_accuracy(exp3_gpu["results"], test_sentiments)

print(f"Sentiment accuracy: {exp3_gpu_acc}")
print(f"Runtime with model loading: {exp3_gpu['runtime_with_load_s']}s")
print(f"Runtime without model loading: {exp3_gpu['runtime_wo_load_s']}s")

Measuring cardiffnlp/twitter-roberta-base-sentiment-latest on GPU...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentiment accuracy: 0.5444
Runtime with model loading: 23.0184s
Runtime without model loading: 21.3009s


## 10. Experiment 4: Domain-adapted Emotion Model (CPU)

Model: the final YouTube-domain adapted model used as the default Streamlit app model.


In [11]:
MODEL_4 = "chase1zhang/youtube-emotion-distilbert-domain-adapted"

print(f"Measuring {MODEL_4} on CPU...")
exp4_cpu = measure_pipeline_runtime(MODEL_4, "text-classification", test_texts, device=-1)
exp4_cpu_acc = compute_emotion_accuracy(exp4_cpu["results"], test_labels)

print(f"Accuracy: {exp4_cpu_acc}")
print(f"Runtime with model loading: {exp4_cpu['runtime_with_load_s']}s")
print(f"Runtime without model loading: {exp4_cpu['runtime_wo_load_s']}s")


Measuring chase1zhang/youtube-emotion-distilbert-domain-adapted on CPU...


config.json:   0%|          | 0.00/853 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Accuracy: 0.4944
Runtime with model loading: 131.0709s
Runtime without model loading: 121.8811s


## 11. Experiment 4: Domain-adapted Emotion Model (GPU)

Run this cell only when the Colab runtime has GPU enabled.


In [12]:
print(f"Measuring {MODEL_4} on GPU...")
exp4_gpu = measure_pipeline_runtime(MODEL_4, "text-classification", test_texts, device=0)
exp4_gpu_acc = compute_emotion_accuracy(exp4_gpu["results"], test_labels)

print(f"Accuracy: {exp4_gpu_acc}")
print(f"Runtime with model loading: {exp4_gpu['runtime_with_load_s']}s")
print(f"Runtime without model loading: {exp4_gpu['runtime_wo_load_s']}s")


Measuring chase1zhang/youtube-emotion-distilbert-domain-adapted on GPU...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Accuracy: 0.4944
Runtime with model loading: 12.0165s
Runtime without model loading: 10.8856s


## 12. Summary: Model Selection Results

Fill this table into the Excel file after running all experiments.


In [13]:
summary_rows = [
    ["j-hartmann/emotion-english-distilroberta-base", "CPU", len(test_texts), exp1_cpu_acc, exp1_cpu['runtime_with_load_s'], exp1_cpu['runtime_wo_load_s']],
    ["j-hartmann/emotion-english-distilroberta-base", "GPU", len(test_texts), exp1_gpu_acc, exp1_gpu['runtime_with_load_s'], exp1_gpu['runtime_wo_load_s']],
    [MODEL_2, "CPU", len(test_texts), exp2_cpu_acc, exp2_cpu['runtime_with_load_s'], exp2_cpu['runtime_wo_load_s']],
    [MODEL_2, "GPU", len(test_texts), exp2_gpu_acc, exp2_gpu['runtime_with_load_s'], exp2_gpu['runtime_wo_load_s']],
    [MODEL_4, "CPU", len(test_texts), exp4_cpu_acc, exp4_cpu['runtime_with_load_s'], exp4_cpu['runtime_wo_load_s']],
    [MODEL_4, "GPU", len(test_texts), exp4_gpu_acc, exp4_gpu['runtime_with_load_s'], exp4_gpu['runtime_wo_load_s']],
    ["cardiffnlp/twitter-roberta-base-sentiment-latest", "CPU", len(test_texts), exp3_cpu_acc, exp3_cpu['runtime_with_load_s'], exp3_cpu['runtime_wo_load_s']],
    ["cardiffnlp/twitter-roberta-base-sentiment-latest", "GPU", len(test_texts), exp3_gpu_acc, exp3_gpu['runtime_with_load_s'], exp3_gpu['runtime_wo_load_s']],
]

summary_df = pd.DataFrame(
    summary_rows,
    columns=["Model", "Device", "Test Samples", "Accuracy", "Runtime with loading (s)", "Runtime w/o loading (s)"]
)
summary_df


,Model,Device,Test Samples,Accuracy,Runtime with loading (s),Runtime w/o loading (s)
0,j-hartmann/emotion-english-distilroberta-base,CPU,2160,0.6204,148.3473,118.4442
1,j-hartmann/emotion-english-distilroberta-base,GPU,2160,0.6204,27.6285,13.8853
2,chase1zhang/youtube-emotion-distilbert,CPU,2160,0.8630,138.9428,122.6187
3,chase1zhang/youtube-emotion-distilbert,GPU,2160,0.8630,11.8892,10.9848
4,chase1zhang/youtube-emotion-distilbert-domain-...,CPU,2160,0.4944,131.0709,121.8811
5,chase1zhang/youtube-emotion-distilbert-domain-...,GPU,2160,0.4944,12.0165,10.8856
6,cardiffnlp/twitter-roberta-base-sentiment-latest,CPU,2160,0.5444,244.9532,241.3229
7,cardiffnlp/twitter-roberta-base-sentiment-latest,GPU,2160,0.5444,23.0184,21.3009


## 13. App Performance Test

The deployed app was tested on three YouTube videos. For each video, 50 comments were manually reviewed comment by comment. A second assistant review revised 23 labels for sarcasm, humor, pride, and threat/concern cues. Accuracy is the number of comments where the model label matches the reviewed manual label divided by 50.


In [14]:
app_test_rows = [
    ["https://www.youtube.com/watch?v=d2dgJGkw5p0", "7-emotion", "pre-tuning baseline", "j-hartmann/emotion-english-distilroberta-base", 50, 29, 0.5800, "neutral", "neutral"],
    ["https://www.youtube.com/watch?v=d2dgJGkw5p0", "7-emotion", "GoEmotions fine-tuned", "chase1zhang/youtube-emotion-distilbert", 50, 29, 0.5800, "neutral", "neutral"],
    ["https://www.youtube.com/watch?v=d2dgJGkw5p0", "7-emotion", "YouTube-domain adapted", "chase1zhang/youtube-emotion-distilbert-domain-adapted", 50, 28, 0.5600, "neutral", "neutral"],
    ["https://www.youtube.com/watch?v=d2dgJGkw5p0", "7-emotion", "Public RoBERTa-large", "j-hartmann/emotion-english-roberta-large", 50, 24, 0.4800, "neutral", "neutral"],
    ["https://www.youtube.com/watch?v=d2dgJGkw5p0", "7-emotion", "Public GoEmotions RoBERTa", "SamLowe/roberta-base-go_emotions", 50, 21, 0.4200, "neutral", "neutral"],
    ["https://www.youtube.com/watch?v=d2dgJGkw5p0", "3-sentiment", "sentiment pipeline", "cardiffnlp/twitter-roberta-base-sentiment-latest", 50, 30, 0.6000, "neutral", "neutral"],
    ["https://www.youtube.com/watch?v=M8To7iorkxQ", "7-emotion", "pre-tuning baseline", "j-hartmann/emotion-english-distilroberta-base", 50, 26, 0.5200, "joy", "neutral"],
    ["https://www.youtube.com/watch?v=M8To7iorkxQ", "7-emotion", "GoEmotions fine-tuned", "chase1zhang/youtube-emotion-distilbert", 50, 25, 0.5000, "joy", "neutral"],
    ["https://www.youtube.com/watch?v=M8To7iorkxQ", "7-emotion", "YouTube-domain adapted", "chase1zhang/youtube-emotion-distilbert-domain-adapted", 50, 25, 0.5000, "joy", "neutral"],
    ["https://www.youtube.com/watch?v=M8To7iorkxQ", "7-emotion", "Public RoBERTa-large", "j-hartmann/emotion-english-roberta-large", 50, 24, 0.4800, "joy", "neutral"],
    ["https://www.youtube.com/watch?v=M8To7iorkxQ", "7-emotion", "Public GoEmotions RoBERTa", "SamLowe/roberta-base-go_emotions", 50, 10, 0.2000, "joy", "neutral"],
    ["https://www.youtube.com/watch?v=M8To7iorkxQ", "3-sentiment", "sentiment pipeline", "cardiffnlp/twitter-roberta-base-sentiment-latest", 50, 46, 0.9200, "positive", "positive"],
    ["https://www.youtube.com/watch?v=-_-eIVAX1yQ", "7-emotion", "pre-tuning baseline", "j-hartmann/emotion-english-distilroberta-base", 50, 12, 0.2400, "anger", "neutral"],
    ["https://www.youtube.com/watch?v=-_-eIVAX1yQ", "7-emotion", "GoEmotions fine-tuned", "chase1zhang/youtube-emotion-distilbert", 50, 16, 0.3200, "anger", "neutral"],
    ["https://www.youtube.com/watch?v=-_-eIVAX1yQ", "7-emotion", "YouTube-domain adapted", "chase1zhang/youtube-emotion-distilbert-domain-adapted", 50, 18, 0.3600, "anger", "neutral"],
    ["https://www.youtube.com/watch?v=-_-eIVAX1yQ", "7-emotion", "Public RoBERTa-large", "j-hartmann/emotion-english-roberta-large", 50, 14, 0.2800, "anger", "neutral"],
    ["https://www.youtube.com/watch?v=-_-eIVAX1yQ", "7-emotion", "Public GoEmotions RoBERTa", "SamLowe/roberta-base-go_emotions", 50, 10, 0.2000, "anger", "neutral"],
    ["https://www.youtube.com/watch?v=-_-eIVAX1yQ", "3-sentiment", "sentiment pipeline", "cardiffnlp/twitter-roberta-base-sentiment-latest", 50, 29, 0.5800, "negative", "negative"],
]

app_test_df = pd.DataFrame(
    app_test_rows,
    columns=["Video", "Task", "Model Stage", "Model", "# Comments", "Matched Comments", "Accuracy", "Manual Main Label", "Model Main Label"],
)
app_test_df

,Video,Task,Model Stage,Model,# Comments,Matched Comments,Accuracy,Manual Main Label,Model Main Label
0,https://www.youtube.com/watch?v=d2dgJGkw5p0,7-emotion,pre-tuning baseline,j-hartmann/emotion-english-distilroberta-base,50,29,0.58,neutral,neutral
1,https://www.youtube.com/watch?v=d2dgJGkw5p0,7-emotion,GoEmotions fine-tuned,chase1zhang/youtube-emotion-distilbert,50,29,0.58,neutral,neutral
2,https://www.youtube.com/watch?v=d2dgJGkw5p0,7-emotion,YouTube-domain adapted,chase1zhang/youtube-emotion-distilbert-domain-...,50,28,0.56,neutral,neutral
3,https://www.youtube.com/watch?v=d2dgJGkw5p0,7-emotion,Public RoBERTa-large,j-hartmann/emotion-english-roberta-large,50,24,0.48,neutral,neutral
4,https://www.youtube.com/watch?v=d2dgJGkw5p0,7-emotion,Public GoEmotions RoBERTa,SamLowe/roberta-base-go_emotions,50,21,0.42,neutral,neutral
5,https://www.youtube.com/watch?v=d2dgJGkw5p0,3-sentiment,sentiment pipeline,cardiffnlp/twitter-roberta-base-sentiment-latest,50,30,0.60,neutral,neutral
6,https://www.youtube.com/watch?v=M8To7iorkxQ,7-emotion,pre-tuning baseline,j-hartmann/emotion-english-distilroberta-base,50,26,0.52,joy,neutral
7,https://www.youtube.com/watch?v=M8To7iorkxQ,7-emotion,GoEmotions fine-tuned,chase1zhang/youtube-emotion-distilbert,50,25,0.50,joy,neutral
8,https://www.youtube.com/watch?v=M8To7iorkxQ,7-emotion,YouTube-domain adapted,chase1zhang/youtube-emotion-distilbert-domain-...,50,25,0.50,joy,neutral
9,https://www.youtube.com/watch?v=M8To7iorkxQ,7-emotion,Public RoBERTa-large,j-hartmann/emotion-english-roberta-large,50,24,0.48,joy,neutral


## 14. Export results to CSV


In [15]:
# Export model comparison results
summary_df.to_csv("model_comparison_results.csv", index=False)
print("Saved model comparison to model_comparison_results.csv")

# Export app test results
app_test_df.to_csv("app_performance_results.csv", index=False)
print("Saved app test results to app_performance_results.csv")

Saved model comparison to model_comparison_results.csv
Saved app test results to app_performance_results.csv
